# Adaptive Query Planning for Multi-Hop QA
This notebook runs the Review-2 comparison from a fresh Colab runtime.

In [ ]:
REPOSITORY_URL = input('Paste your GitHub repository URL: ').strip()
if not REPOSITORY_URL: raise ValueError('A repository URL is required.')
!git clone $REPOSITORY_URL
repo_dir = REPOSITORY_URL.rstrip('/').split('/')[-1].removesuffix('.git')
%cd $repo_dir
!pip -q install -r requirements.txt

In [ ]:
# Optional: enable the configured OpenAI-compatible LLM provider with a Colab Secret.
from google.colab import userdata
import os
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    USE_LLM = True
except Exception:
    USE_LLM = False
print('LLM enabled:', USE_LLM)

In [ ]:
import yaml
from pathlib import Path
config = yaml.safe_load(Path('config.yaml').read_text())
config['dataset'] = {'name': 'hotpotqa', 'split': 'validation', 'sample_size': 10}
config['llm']['enabled'] = USE_LLM
config['output']['directory'] = 'results'
Path('colab_config.yaml').write_text(yaml.safe_dump(config, sort_keys=False))

In [ ]:
!python -m pytest tests -q

In [ ]:
!python experiments/run_experiment.py --config colab_config.yaml

In [ ]:
import glob, pandas as pd
run_dir = sorted(glob.glob('results/*'))[-1]
display(pd.read_csv(f'{run_dir}/comparison.csv'))
from IPython.display import Image, display
display(Image(f'{run_dir}/comparison.png'))
print('Raw records and summaries:', run_dir)